In [2]:
# Load the CMS Part D spending data into a table and print them out in rows and columns
import duckdb

billion = 1000000000
connection = duckdb.connect()

query = """
SELECT *
FROM read_csv_auto('../data/partd_raw.csv', header = true, all_varchar = true)
"""

data = connection.execute(query).df()

print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))

Number of rows: 14536
Number of columns: 46


In [3]:
# List every column to know what the file contains
for column_name in data.columns:
    print(column_name)

Brnd_Name
Gnrc_Name
Tot_Mftr
Mftr_Name
Tot_Spndng_2020
Tot_Dsg_Unts_2020
Tot_Clms_2020
Tot_Benes_2020
Avg_Spnd_Per_Dsg_Unt_Wghtd_2020
Avg_Spnd_Per_Clm_2020
Avg_Spnd_Per_Bene_2020
Outlier_Flag_2020
Tot_Spndng_2021
Tot_Dsg_Unts_2021
Tot_Clms_2021
Tot_Benes_2021
Avg_Spnd_Per_Dsg_Unt_Wghtd_2021
Avg_Spnd_Per_Clm_2021
Avg_Spnd_Per_Bene_2021
Outlier_Flag_2021
Tot_Spndng_2022
Tot_Dsg_Unts_2022
Tot_Clms_2022
Tot_Benes_2022
Avg_Spnd_Per_Dsg_Unt_Wghtd_2022
Avg_Spnd_Per_Clm_2022
Avg_Spnd_Per_Bene_2022
Outlier_Flag_2022
Tot_Spndng_2023
Tot_Dsg_Unts_2023
Tot_Clms_2023
Tot_Benes_2023
Avg_Spnd_Per_Dsg_Unt_Wghtd_2023
Avg_Spnd_Per_Clm_2023
Avg_Spnd_Per_Bene_2023
Outlier_Flag_2023
Tot_Spndng_2024
Tot_Dsg_Unts_2024
Tot_Clms_2024
Tot_Benes_2024
Avg_Spnd_Per_Dsg_Unt_Wghtd_2024
Avg_Spnd_Per_Clm_2024
Avg_Spnd_Per_Bene_2024
Outlier_Flag_2024
Chg_Avg_Spnd_Per_Dsg_Unt_23_24
CAGR_Avg_Spnd_Per_Dsg_Unt_20_24


In [4]:
# Register data as a table and check which brand and generic drug appear more than once
connection.register("drugs", data)

query = """
SELECT
    Brnd_Name,
    Gnrc_Name,
    COUNT(*) AS row_count
FROM drugs
GROUP BY Brnd_Name, Gnrc_Name
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10
"""

connection.execute(query).df()

,Brnd_Name,Gnrc_Name,row_count
0,Potassium Chloride*,Potassium Chloride,47
1,Gabapentin,Gabapentin,41
2,Levetiracetam*,Levetiracetam,40
3,Acyclovir*,Acyclovir,38
4,Pantoprazole Sodium*,Pantoprazole Sodium,37
5,Famotidine*,Famotidine,34
6,Nystatin*,Nystatin,32
7,Atorvastatin Calcium,Atorvastatin Calcium,31
8,Ketorolac Tromethamine*,Ketorolac Tromethamine,31
9,Doxycycline Hyclate*,Doxycycline Hyclate,31


In [5]:
# Gives the number of distinct drugs
query = """
SELECT COUNT(*) AS distinct_drugs
FROM drugs
WHERE Mftr_Name = 'Overall'
"""

connection.execute(query).df()

,distinct_drugs
0,3625


In [6]:
# Creates a new table that organizes drug spending, claims, dosage units, and beneficiaries by year from 2020 to 2024

query = """
CREATE OR REPLACE TABLE spending_by_year AS

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2020 AS year,
    TRY_CAST(Tot_Spndng_2020   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2020 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2020     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2020    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2021 AS year,
    TRY_CAST(Tot_Spndng_2021   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2021 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2021     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2021    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2022 AS year,
    TRY_CAST(Tot_Spndng_2022   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2022 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2022     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2022    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2023 AS year,
    TRY_CAST(Tot_Spndng_2023   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2023 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2023     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2023    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'

UNION ALL

SELECT
    Brnd_Name AS brand_name,
    Gnrc_Name AS generic_name,
    2024 AS year,
    TRY_CAST(Tot_Spndng_2024   AS DOUBLE) AS spending,
    TRY_CAST(Tot_Dsg_Unts_2024 AS DOUBLE) AS dosage_units,
    TRY_CAST(Tot_Clms_2024     AS DOUBLE) AS claims,
    TRY_CAST(Tot_Benes_2024    AS DOUBLE) AS beneficiaries
FROM drugs
WHERE Mftr_Name = 'Overall'
"""

connection.execute(query)

print("Table created.")

Table created.


In [7]:
# Counts how many rows of drug data are included for each year

query = """
SELECT
    year,
    COUNT(*) AS row_count
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,row_count
0,2020,3625
1,2021,3625
2,2022,3625
3,2023,3625
4,2024,3625


In [8]:
# Calculates total drug spending for each year and shows how many rows have spending data

query = """
SELECT
    year,
    SUM(spending) / 1000000000 AS spending_billions,
    COUNT(*) AS total_rows,
    COUNT(spending) AS rows_with_value
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,spending_billions,total_rows,rows_with_value
0,2020,194.092038,3625,2887
1,2021,213.513349,3625,3054
2,2022,239.779216,3625,3221
3,2023,275.808919,3625,3405
4,2024,288.667842,3625,3625


In [9]:
# Calculates yearly drug spending, dosage units, and the average spending per dosage unit

query = """
SELECT
    year,
    SUM(spending) / 1000000000 AS spending_billions,
    SUM(dosage_units) / 1000000000 AS units_billions,
    SUM(spending) / SUM(dosage_units) AS price_per_unit
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,spending_billions,units_billions,price_per_unit
0,2020,194.092038,110.824788,1.751341
1,2021,213.513349,113.394110,1.882932
2,2022,239.779216,119.426392,2.007757
3,2023,275.808919,124.660472,2.212481
4,2024,288.667842,131.212311,2.200006


In [10]:
# Pull the first and last year to compare them
query = """
SELECT
    year,
    SUM(spending) AS total_spending,
    SUM(dosage_units) AS total_units
FROM spending_by_year
WHERE year IN (2020, 2024)
GROUP BY year
ORDER BY year
"""

totals = connection.execute(query).df()
totals

,year,total_spending,total_units
0,2020,1.940920e+11,1.108248e+11
1,2024,2.886678e+11,1.312123e+11


In [11]:
# Read the two rows from the table above
# Sorted by year
# Row 0 is 2020 and row 1 is 2024
spending_2020 = totals["total_spending"][0]
spending_2024 = totals["total_spending"][1]
units_2020 = totals["total_units"][0]
units_2024 = totals["total_units"][1]

# Price is spending divided by units
price_2020 = spending_2020 / units_2020
price_2024 = spending_2024 / units_2024

# Price effect: prices changed when volume held at 2020 levels
price_effect = (price_2024 - price_2020) * units_2020

# Volume effect: volume changed when prices held at 2020 levels
volume_effect = (units_2024 - units_2020) * price_2020

# Interaction: the extra from both changing at once
interaction = (price_2024 - price_2020) * (units_2024 - units_2020)

total_change = spending_2024 - spending_2020

print("Total change:  ", round(total_change / billion, 1), "billion")
print("Price effect:  ", round(price_effect / billion, 1), "billion")
print("Volume effect: ", round(volume_effect / billion, 1), "billion")
print("Interaction:   ", round(interaction / billion, 1), "billion")

Total change:   94.6 billion
Price effect:   49.7 billion
Volume effect:  35.7 billion
Interaction:    9.1 billion


In [12]:
# Shows 2020 and 2024 values next to each other
# CASE WHEN checks which year the value belongs to

query = """
CREATE OR REPLACE TABLE drug_growth AS
SELECT
    brand_name,
    generic_name,
    SUM(CASE WHEN year = 2020 THEN spending ELSE 0 END) AS spending_2020,
    SUM(CASE WHEN year = 2024 THEN spending ELSE 0 END) AS spending_2024,
    SUM(CASE WHEN year = 2020 THEN dosage_units ELSE 0 END) AS units_2020,
    SUM(CASE WHEN year = 2024 THEN dosage_units ELSE 0 END) AS units_2024
FROM spending_by_year
GROUP BY brand_name, generic_name
"""

connection.execute(query)

print("Table created")

Table created


In [13]:
# Shows the 15 drugs where spending increased the most from 2020 to 2024
query = """
SELECT
    brand_name,
    spending_2020 / 1000000 AS spending_2020_millions,
    spending_2024 / 1000000 AS spending_2024_millions,
    (spending_2024 - spending_2020) / 1000000 AS growth_millions
FROM drug_growth
ORDER BY growth_millions DESC
LIMIT 15
"""
connection.execute(query).df()

,brand_name,spending_2020_millions,spending_2024_millions,growth_millions
0,Ozempic,1455.812267,12970.296347,11514.484080
1,Eliquis,9936.069814,20774.929225,10838.859411
2,Jardiance,2376.166293,11435.997421,9059.831128
3,Mounjaro,0.000000,6336.942641,6336.942641
4,Farxiga,736.787564,5290.548963,4553.761400
5,Trelegy Ellipta,1487.802308,5294.401777,3806.599469
6,Entresto,1203.043540,4020.321686,2817.278146
7,Lenalidomide,0.000000,2640.854800,2640.854800
8,Vyndamax,253.267541,2830.811857,2577.544317
9,Stelara*,1106.356248,3378.017374,2271.661126


In [14]:
# Total growth across all drugs, 2020 to 2024
query = """
SELECT
    SUM(spending_2024 - spending_2020) / 1000000000 AS total_growth_billions
FROM drug_growth
"""

total = connection.execute(query).df()
total

,total_growth_billions
0,94.575804


In [15]:
# Growth from the 15 largest contributors
query = """
SELECT SUM(growth) / 1000000000 AS top15_growth_billions
FROM (
    SELECT spending_2024 - spending_2020 AS growth
    FROM drug_growth
    ORDER BY growth DESC
    LIMIT 15
)
"""

top15 = connection.execute(query).df()

share = top15["top15_growth_billions"][0] / total["total_growth_billions"][0]

print("Top 15 growth:", round(top15["top15_growth_billions"][0], 1), "billion")
print("Total growth: ", round(total["total_growth_billions"][0], 1), "billion")
print("Share:        ", round(share * 100, 1), "percent")

Top 15 growth: 66.1 billion
Total growth:  94.6 billion
Share:         69.9 percent


In [ ]:
# Lenalidomide is the generic (cheaper) of the brand-name drug Revlimid
# Some patients switched from Revlimid to Lenalidomide
# Spending alone can't tell us if more patients are taking the drug,
# or if the price went up. So we also look at units (number of pills)
# and price per unit, for both the brand and the generic
query = """
SELECT
    brand_name,
    ROUND(spending_2020 / 1000000, 1) AS spending_2020_millions,
    ROUND(spending_2024 / 1000000, 1) AS spending_2024_millions,
    ROUND(units_2020 / 1000000, 2) AS units_2020_millions,
    ROUND(units_2024 / 1000000, 2) AS units_2024_millions,
    ROUND(spending_2020 / units_2020, 2) AS price_per_unit_2020,
    ROUND(spending_2024 / units_2024, 2) AS price_per_unit_2024
FROM drug_growth
WHERE brand_name LIKE '%Revlimid%'
   OR brand_name LIKE '%Lenalidomide%'
"""

revlimid = connection.execute(query).df()
display(revlimid)

# Add the brand and generic together to see the molecule as one drug
units_2020_combined = revlimid["units_2020_millions"].sum()
units_2024_combined = revlimid["units_2024_millions"].sum()

print("Combined units 2020:", round(units_2020_combined, 2), "million")
print("Combined units 2024:", round(units_2024_combined, 2), "million")
print("Change:             ", round(units_2024_combined - units_2020_combined, 2), "million")

,brand_name,spending_2020_millions,spending_2024_millions,units_2020_millions,units_2024_millions,price_per_unit_2020,price_per_unit_2024
0,Lenalidomide,0.0,2640.9,0.00,4.03,NaN,655.95
1,Revlimid,5356.1,4179.5,6.74,4.89,794.31,854.90


Combined units 2020: 6.74 million
Combined units 2024: 8.92 million
Change:              2.18 million


In [ ]:
# Some drugs are listed under more than one name
# For example, Victoza is listed as "Victoza 2-Pak" and "Victoza 3-Pak"
# So searching for the exact name "Victoza" finds nothing
# The next cell searches by the generic name instead, so it finds every version
query = """
SELECT
    brand_name,
    generic_name,
    ROUND(spending_2024 / 1000000, 1) AS spending_2024_millions
FROM drug_growth
WHERE generic_name LIKE '%liraglutide%'
   OR brand_name LIKE '%Victoza%'
"""

connection.execute(query).df()

,brand_name,generic_name,spending_2024_millions
0,Victoza 3-Pak,Liraglutide,203.1
1,Victoza 2-Pak,Liraglutide,77.8


In [ ]:
# Match on the molecule rather than the brand. CMS splits brand names
# by package size ("Victoza 2-Pak"), so exact brand matching misses rows.
query = """
SELECT
    brand_name,
    generic_name,
    ROUND(spending_2020 / 1000000, 1) AS spending_2020_millions,
    ROUND(spending_2024 / 1000000, 1) AS spending_2024_millions,
    ROUND((spending_2024 - spending_2020) / 1000000, 1) AS growth_millions
FROM drug_growth
WHERE generic_name ILIKE '%semaglutide%'
   OR generic_name ILIKE '%tirzepatide%'
   OR generic_name ILIKE '%dulaglutide%'
   OR generic_name ILIKE '%liraglutide%'
   OR generic_name ILIKE '%exenatide%'
ORDER BY growth_millions DESC
"""

glp1 = connection.execute(query).df()
display(glp1)

glp1_growth = glp1["growth_millions"].sum() / 1000
share = glp1_growth / 94.6

print("Rows found:", len(glp1))
print("GLP-1 growth:", round(glp1_growth, 2), "billion")
print("Share of total growth:", round(share * 100, 1), "percent")

,brand_name,generic_name,spending_2020_millions,spending_2024_millions,growth_millions
0,Ozempic,Semaglutide,1455.8,12970.3,11514.5
1,Mounjaro,Tirzepatide,0.0,6336.9,6336.9
2,Trulicity,Dulaglutide,3284.9,5454.1,2169.3
3,Rybelsus,Semaglutide,73.4,1890.1,1816.6
4,Wegovy,Semaglutide,0.0,301.3,301.3
5,Liraglutide,Liraglutide,0.0,48.1,48.1
6,Xultophy 100-3.6,Insulin Degludec/Liraglutide,71.9,91.3,19.3
7,Zepbound,Tirzepatide,0.0,0.4,0.4
8,Byetta,Exenatide,47.7,15.9,-31.8
9,Bydureon Bcise,Exenatide Microspheres,266.2,215.4,-50.8


Rows found: 12
GLP-1 growth: 20.51 billion
Share of total growth: 21.7 percent


In [ ]:
# The table should hold exactly one row per drug per year
# Anything returned here means totals are inflated
query = """
SELECT
    brand_name,
    generic_name,
    year,
    COUNT(*) AS row_count
FROM spending_by_year
GROUP BY brand_name, generic_name, year
HAVING COUNT(*) > 1
"""

duplicates = connection.execute(query).df()

print("Duplicate drug-year rows:", len(duplicates))

Duplicate drug-year rows: 0


In [ ]:
# Spending cannot be negative, and spending with no units implies
# a price that cannot exist
query = """
SELECT
    year,
    SUM(CASE WHEN spending < 0 THEN 1 ELSE 0 END) AS negative_spending,
    SUM(CASE WHEN dosage_units < 0 THEN 1 ELSE 0 END) AS negative_units,
    SUM(CASE WHEN dosage_units = 0 AND spending > 0 THEN 1 ELSE 0 END) AS spending_no_units
FROM spending_by_year
GROUP BY year
ORDER BY year
"""

connection.execute(query).df()

,year,negative_spending,negative_units,spending_no_units
0,2020,0.0,0.0,0.0
1,2021,0.0,0.0,0.0
2,2022,0.0,0.0,0.0
3,2023,0.0,0.0,0.0
4,2024,0.0,0.0,0.0


In [ ]:
# spending_by_year and drug_growth are built separately
# Their 2024 totals should be identical
query = """
SELECT SUM(spending) / 1000000000 AS total_billions
FROM spending_by_year
WHERE year = 2024
"""
program_level = connection.execute(query).df()["total_billions"][0]

query = """
SELECT SUM(spending_2024) / 1000000000 AS total_billions
FROM drug_growth
"""
drug_level = connection.execute(query).df()["total_billions"][0]

print("Program level 2024:", round(program_level, 3), "billion")
print("Drug level 2024:   ", round(drug_level, 3), "billion")
print("Difference:        ", round(program_level - drug_level, 6), "billion")

Program level 2024: 288.668 billion
Drug level 2024:    288.668 billion
Difference:         0.0 billion


In [ ]:
# Spending = price x volume, so spending growth should equal
# price growth multiplied by volume growth
query = """
SELECT
    SUM(CASE WHEN year = 2020 THEN spending     END) AS spending_2020,
    SUM(CASE WHEN year = 2024 THEN spending     END) AS spending_2024,
    SUM(CASE WHEN year = 2020 THEN dosage_units END) AS units_2020,
    SUM(CASE WHEN year = 2024 THEN dosage_units END) AS units_2024
FROM spending_by_year
"""

endpoints = connection.execute(query).df()

spending_2020 = endpoints["spending_2020"][0]
spending_2024 = endpoints["spending_2024"][0]
units_2020 = endpoints["units_2020"][0]
units_2024 = endpoints["units_2024"][0]

price_2020 = spending_2020 / units_2020
price_2024 = spending_2024 / units_2024

price_growth = price_2024 / price_2020
volume_growth = units_2024 / units_2020
spending_growth = spending_2024 / spending_2020

print("Price growth:   ", round(price_growth, 5))
print("Volume growth:  ", round(volume_growth, 5))
print("Product:        ", round(price_growth * volume_growth, 5))
print("Spending growth:", round(spending_growth, 5))
print("Gap:            ", round(price_growth * volume_growth - spending_growth, 8))

Price growth:    1.25618
Volume growth:   1.18396
Product:         1.48727
Spending growth: 1.48727
Gap:             -0.0


In [ ]:
# CMS marks some drugs as "outliers". This means their price per unit
# may be wrong because of unusual records in the data
# The analysis depends on price per unit, so we check how much
# spending these drugs make up. If it's small, they don't change our results.
#   Outlier_Flag_2024 = 1 means flagged, 0 means not flagged,
#   and blank (NaN) means CMS didn't give a flag
query = """
SELECT
    Outlier_Flag_2024,
    COUNT(*) AS drug_count,
    ROUND(SUM(TRY_CAST(Tot_Spndng_2024 AS DOUBLE)) / 1000000000, 2) AS spending_billions
FROM drugs
WHERE Mftr_Name = 'Overall'
GROUP BY Outlier_Flag_2024
ORDER BY Outlier_Flag_2024
"""

connection.execute(query).df()

,Outlier_Flag_2024,drug_count,spending_billions
0,0,3212,286.20
1,1,395,2.47
2,NaN,18,0.00
